# Preprocess breath-by-breath and beat-by-beat data

Use the [Windows/Python 3.7.4 kernel](../environments/README.md). Launch Jupyter from the repository root or `notebooks/` and run cells top to bottom.

Configure paths below, set `run_preprocessing = True`, and optionally enable `plot_diagnostics`. Both switches default to disabled.

In [ ]:
import hashlib
import json
import os
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from scipy.ndimage import median_filter
from scipy.signal import butter, filtfilt, find_peaks, iirnotch, savgol_filter

## Inputs

Set `AI_CPET_RAW_DATA_DIR` (default `data/raw`) and `AI_CPET_PREPROCESSED_DIR` (default `data/preprocessed`). Relative paths start at the repository root. Use a new output directory outside the input tree; neither directory may contain the other.

For each `<cohort>` (`control` or `ots`), provide:

| File | Required contents |
| --- | --- |
| `<cohort>_participants.xlsx` | Only `testID`, in processing order. Unique string IDs across both groups: letters/digits/underscores/hyphens, starting with a letter or digit. |
| `<cohort>_test_times.xlsx` | `testID`, `load_start_time`, `exhaustion_time`; one row per participant. Elapsed `HH:MM:SS` strings; exhaustion follows load start. |
| `<cohort>_bbb/<testID>.xlsx` | [BBB columns in order](../README.md#private-data-and-paths), but raw `time` must be `HH:MM:SS` or `MM:SS` strings, not numeric seconds. Preserve row order and units. |
| `<cohort>_ecg/<testID>.adicht` | 1000 Hz two-channel recordings; signal channels first. Verify channel order, recording blocks, and units. |
| `<cohort>_clinical.xlsx` | `testID`, `age`, `height`, `sex`, `weight`, in order; sex is `male` or `female`. |
| `<cohort>_cpet.xlsx` | `testID` and [eleven CPET features in order](../README.md#private-data-and-paths); preserve units. |

## Outputs

Review failures in `run-report.json`; never use an incomplete run. Single-channel recordings are not exported. Output tables include only participants processed successfully in both recording types. Inputs are never overwritten or deleted.

Set `AI_CPET_DATA_DIR` to the absolute output directory before imports in the Python 3.11 model kernel. Verify participant membership and order across all four data types. Keep inputs and results private.

## Preprocessing functions

In [ ]:
def filter_raw_signal(signal):
    fs = 1000
    f0 = 50.0
    Q = 30
    b, a = iirnotch(f0, Q, fs)
    filtered_signal = filtfilt(b, a, signal)
    nyq = 0.5 * fs
    low = 0.5 / nyq
    high = 100 / nyq
    order = 2
    b, a = butter(order, [low, high], btype="band")
    filtered_signal = filtfilt(b, a, filtered_signal)
    return filtered_signal

In [ ]:
def iron_out_spikes(signal):
    peaks, _ = find_peaks(signal, prominence=30)
    for peak in peaks:
        start_of_peak = int(peak - 10)
        end_of_peak = int(peak + 10)
        ironed_part = median_filter(signal[start_of_peak:end_of_peak], size=20, mode="nearest")
        signal[start_of_peak:end_of_peak] = ironed_part
    return signal

In [ ]:
def compute_HR_signal(f):
    fs = 1000
    ecg1 = []
    ecg2 = []
    try:
        if f.channels[0].n_records == 1:
            ecg1 = f.channels[0].get_data(1)
            ecg2 = f.channels[1].get_data(1)
        else:
            for n in range(f.channels[0].n_records):
                if n > 0:
                    if len(ecg1) < 1500000.0:
                        ecg1 = np.append(ecg1, f.channels[0].get_data(n + 1))
                    if len(ecg2) < 1500000.0:
                        ecg2 = np.append(ecg2, f.channels[1].get_data(n + 1))
                else:
                    ecg1 = np.append(ecg1, f.channels[0].get_data(n + 1))
                    ecg2 = np.append(ecg2, f.channels[1].get_data(n + 1))
        use_both_ecg = True
        var_ecg1 = np.var(ecg1)
        var_ecg2 = np.var(ecg2)
    except:
        if f.channels[0].n_records == 1:
            ecg1 = f.channels[0].get_data(1)
        else:
            for n in range(f.channels[0].n_records):
                ecg1 = np.append(ecg1, f.channels[0].get_data(n + 1))
        use_both_ecg = False
        var_ecg1 = np.var(ecg1)
        var_ecg2 = 99
    n_samples = len(ecg1)
    time = np.linspace(0, n_samples / fs, n_samples)
    if var_ecg1 / var_ecg2 > 3 or var_ecg1 / var_ecg2 < 1 / 3:
        use_both_ecg = False
    if use_both_ecg:
        ecg1 = filter_raw_signal(ecg1)
        ecg2 = filter_raw_signal(ecg2)
        v = np.linspace(0.5 * np.pi, 1.5 * np.pi, 30)
        peak_filter = np.sin(v)
        ecg1 = np.correlate(ecg1, peak_filter, mode="same")
        ecg2 = np.correlate(ecg2, peak_filter, mode="same")
        ecg1 = (ecg1 - np.mean(ecg1)) / np.std(ecg1)
        ecg2 = (ecg2 - np.mean(ecg2)) / np.std(ecg2)
        threshold = 0.8
        peaks1, _ = find_peaks(ecg1, height=threshold, distance=270)
        peaks2, _ = find_peaks(ecg2, height=threshold, distance=270)
        hr1 = 60 / np.gradient(peaks1) * fs
        hr2 = 60 / np.gradient(peaks2) * fs
        hr1_filtered = iron_out_spikes(hr1)
        hr2_filtered = iron_out_spikes(hr2)
        hr1_filtered = savgol_filter(hr1_filtered, 101, 6)
        hr2_filtered = savgol_filter(hr2_filtered, 101, 6)
        hr1_interp_func = interp1d(
            time[peaks1], hr1_filtered, kind="next", fill_value="extrapolate"
        )
        hr1_inter = hr1_interp_func(time)
        hr2_interp_func = interp1d(
            time[peaks2], hr2_filtered, kind="next", fill_value="extrapolate"
        )
        hr2_inter = hr2_interp_func(time)
        hr_signal = (hr1_inter + hr2_inter) / 2
    else:
        if len(ecg2) == 0:
            ecg = filter_raw_signal(ecg1)
        else:
            mean_ratio = np.mean(abs(ecg1)) / np.mean(abs(ecg2))
            if mean_ratio > 0.2 and mean_ratio < 5:
                ecg = ecg1 if var_ecg1 < var_ecg2 else ecg2
            else:
                ecg = ecg1 if np.mean(abs(ecg1)) < np.mean(abs(ecg2)) else ecg2
        v = np.linspace(0.5 * np.pi, 1.5 * np.pi, 30)
        peak_filter = np.sin(v)
        ecg = np.correlate(ecg, peak_filter, mode="same")
        ecg = (ecg - np.mean(ecg)) / np.std(ecg)
        threshold = 0.8
        peaks, _ = find_peaks(ecg, height=threshold, distance=250)
        hr = 60 / np.gradient(peaks) * fs
        hr_filtered = iron_out_spikes(hr)
        hr_filtered = savgol_filter(hr_filtered, 101, 6)
        hr_interp_func = interp1d(time[peaks], hr_filtered, kind="next", fill_value="extrapolate")
        hr_signal = hr_interp_func(time)
    return (time, hr_signal)

In [ ]:
def return_raw_ecg(f):
    ecg1 = []
    ecg2 = []
    try:
        if f.channels[0].n_records == 1:
            ecg1 = f.channels[0].get_data(1)
            ecg2 = f.channels[1].get_data(1)
        else:
            for n in range(f.channels[0].n_records):
                if n > 0:
                    if len(ecg1) < 1500000.0:
                        ecg1 = np.append(ecg1, f.channels[0].get_data(n + 1))
                    if len(ecg2) < 1500000.0:
                        ecg2 = np.append(ecg2, f.channels[1].get_data(n + 1))
                else:
                    ecg1 = np.append(ecg1, f.channels[0].get_data(n + 1))
                    ecg2 = np.append(ecg2, f.channels[1].get_data(n + 1))
    except:
        if f.channels[0].n_records == 1:
            ecg1 = f.channels[0].get_data(1)
        else:
            for n in range(f.channels[0].n_records):
                ecg1 = np.append(ecg1, f.channels[0].get_data(n + 1))
    return (ecg1, ecg2)

In [ ]:
def detect_end_of_test(hr_signal, time):
    trim_coef = 0.45
    half_hr = hr_signal[round(len(hr_signal) * trim_coef) :]
    half_time = time[round(len(hr_signal) * trim_coef) :]
    window = 30000
    overlap = 15000
    n_windows = len(half_hr) // window
    slope_list = np.array([None, None, None])
    slope_threshold = -0.15
    desperate = False

    def is_end_of_test(win, x):
        try:
            A = np.vstack([x, np.ones(len(x))]).T
            m, _ = np.linalg.lstsq(A, win, rcond=None)[0]
        except:
            m = 0
        return m

    for i in range(n_windows + 1):
        if i != n_windows:
            win = half_hr[i * window : (i + 1) * window]
            x = half_time[i * window : (i + 1) * window]
        else:
            win = half_hr[i * window : len(hr_signal)]
            x = half_time[i * window : len(hr_signal)]
        m = is_end_of_test(win, x)
        if m < slope_threshold and m > -1.75:
            win_next = half_hr[(i + 1) * window - overlap : (i + 2) * window - overlap]
            x_next = half_time[(i + 1) * window - overlap : (i + 2) * window - overlap]
            m_next = is_end_of_test(win_next, x_next)
            if m_next < slope_threshold / 2 and len(x_next) == 30000:
                win_next_next = half_hr[
                    (i + 2) * window - 2 * overlap : (i + 3) * window - 2 * overlap
                ]
                x_next_next = half_time[
                    (i + 2) * window - 2 * overlap : (i + 3) * window - 2 * overlap
                ]
                m_next_next = is_end_of_test(win_next_next, x_next_next)
                if m_next_next < 0 and len(x_next_next) == 30000:
                    if np.mean(win) > np.mean(win_next) > np.mean(win_next_next):
                        slope_list = np.vstack(
                            [slope_list, [m, np.argmax(win) + i * window, np.max(win)]]
                        )
    if slope_list.shape == (3,):
        for i in range(n_windows + 1):
            if i != n_windows:
                win = half_hr[i * window : (i + 1) * window]
                x = half_time[i * window : (i + 1) * window]
            else:
                win = half_hr[i * window : len(hr_signal)]
                x = half_time[i * window : len(hr_signal)]
            m = is_end_of_test(win, x)
            if m < slope_threshold and m > -3:
                win_next = half_hr[(i + 1) * window - overlap : (i + 2) * window - overlap]
                x_next = half_time[(i + 1) * window - overlap : (i + 2) * window - overlap]
                m_next = is_end_of_test(win_next, x_next)
                if m_next < slope_threshold / 2 and len(x_next) == 30000:
                    slope_list = np.vstack(
                        [slope_list, [m, np.argmax(win) + i * window, np.max(win)]]
                    )
    if slope_list.shape == (3,):
        for i in range(n_windows + 1):
            if i != n_windows:
                win = half_hr[i * window : (i + 1) * window]
                x = half_time[i * window : (i + 1) * window]
            else:
                win = half_hr[i * window : len(hr_signal)]
                x = half_time[i * window : len(hr_signal)]
            m = is_end_of_test(win, x)
            if m < slope_threshold / 3 and m > -3:
                desperate = True
                slope_list = np.vstack([slope_list, [m, np.argmax(win) + i * window, max(win)]])
    if desperate:
        end_of_test_idx = round(slope_list[slope_list.shape[0] - 1, 1])
    else:
        slope_list_max = np.argmax(slope_list[1:, 2]) + 1
        end_of_test_idx = round(slope_list[slope_list_max, 1])
    offset = 5000
    grad_threshold = -0.0001
    for i in range(60):
        win_end = end_of_test_idx
        current_hr_max = half_hr[end_of_test_idx]
        offset_val_1 = np.max(half_hr[win_end - 1 * offset : win_end])
        offset_val_2 = np.max(half_hr[win_end - 2 * offset : win_end - 1 * offset])
        offset_val_3 = np.max(half_hr[win_end - 3 * offset : win_end - 2 * offset])
        offset_val_4 = np.max(half_hr[win_end - 4 * offset : win_end - 3 * offset])
        offset_val_5 = np.max(half_hr[win_end - 5 * offset : win_end - 4 * offset])
        offset_val_6 = np.max(half_hr[win_end - 6 * offset : win_end - 5 * offset])
        offset_val_7 = np.max(half_hr[win_end - 7 * offset : win_end - 6 * offset])
        offset_val_8 = np.max(half_hr[win_end - 8 * offset : win_end - 7 * offset])
        offset_val_9 = np.max(half_hr[win_end - 9 * offset : win_end - 8 * offset])
        offset_val_10 = np.max(half_hr[win_end - 10 * offset : win_end - 9 * offset])
        offset_val_11 = np.max(half_hr[win_end - 11 * offset : win_end - 10 * offset])
        offset_val_12 = np.max(half_hr[win_end - 12 * offset : win_end - 11 * offset])
        win1 = half_hr[win_end - 1 * offset : win_end]
        win2 = half_hr[win_end - 2 * offset : win_end - 1 * offset]
        win3 = half_hr[win_end - 3 * offset : win_end - 2 * offset]
        win4 = half_hr[win_end - 4 * offset : win_end - 3 * offset]
        win5 = half_hr[win_end - 5 * offset : win_end - 4 * offset]
        win6 = half_hr[win_end - 6 * offset : win_end - 5 * offset]
        win7 = half_hr[win_end - 7 * offset : win_end - 6 * offset]
        win8 = half_hr[win_end - 8 * offset : win_end - 7 * offset]
        win9 = half_hr[win_end - 9 * offset : win_end - 8 * offset]
        win10 = half_hr[win_end - 10 * offset : win_end - 9 * offset]
        win11 = half_hr[win_end - 11 * offset : win_end - 10 * offset]
        win12 = half_hr[win_end - 12 * offset : win_end - 11 * offset]
        offset_grad_1 = np.mean(np.gradient(half_hr[win_end - 1 * offset : win_end]))
        offset_grad_2 = np.mean(np.gradient(half_hr[win_end - 2 * offset : win_end - 1 * offset]))
        offset_grad_3 = np.mean(np.gradient(half_hr[win_end - 3 * offset : win_end - 2 * offset]))
        offset_grad_4 = np.mean(np.gradient(half_hr[win_end - 4 * offset : win_end - 3 * offset]))
        offset_grad_5 = np.mean(np.gradient(half_hr[win_end - 5 * offset : win_end - 4 * offset]))
        offset_grad_6 = np.mean(np.gradient(half_hr[win_end - 6 * offset : win_end - 5 * offset]))
        offset_grad_7 = np.mean(np.gradient(half_hr[win_end - 7 * offset : win_end - 6 * offset]))
        offset_grad_8 = np.mean(np.gradient(half_hr[win_end - 8 * offset : win_end - 7 * offset]))
        offset_grad_9 = np.mean(np.gradient(half_hr[win_end - 9 * offset : win_end - 8 * offset]))
        offset_grad_10 = np.mean(np.gradient(half_hr[win_end - 10 * offset : win_end - 9 * offset]))
        offset_grad_11 = np.mean(
            np.gradient(half_hr[win_end - 11 * offset : win_end - 10 * offset])
        )
        offset_grad_12 = np.mean(
            np.gradient(half_hr[win_end - 12 * offset : win_end - 11 * offset])
        )
        if offset_grad_1 < grad_threshold and offset_val_1 > current_hr_max:
            end_of_test_idx = end_of_test_idx - 1 * offset - np.argmax(win1)
        elif offset_grad_2 < grad_threshold and offset_val_2 > current_hr_max:
            end_of_test_idx = end_of_test_idx - 2 * offset - np.argmax(win2)
        elif offset_grad_3 < grad_threshold and offset_val_3 > current_hr_max:
            end_of_test_idx = end_of_test_idx - 3 * offset - np.argmax(win3)
        elif offset_grad_4 < grad_threshold and offset_val_4 > current_hr_max:
            end_of_test_idx = end_of_test_idx - 4 * offset - np.argmax(win4)
        elif offset_grad_5 < grad_threshold and offset_val_5 > current_hr_max:
            end_of_test_idx = end_of_test_idx - 5 * offset - np.argmax(win5)
        elif offset_grad_6 < grad_threshold and offset_val_6 > current_hr_max:
            end_of_test_idx = end_of_test_idx - 6 * offset - np.argmax(win6)
        elif offset_grad_7 < grad_threshold and offset_val_7 > current_hr_max:
            end_of_test_idx = end_of_test_idx - 7 * offset - np.argmax(win7)
        elif offset_grad_8 < grad_threshold and offset_val_8 > current_hr_max:
            end_of_test_idx = end_of_test_idx - 8 * offset - np.argmax(win8)
        elif offset_grad_9 < grad_threshold and offset_val_9 > current_hr_max:
            end_of_test_idx = end_of_test_idx - 9 * offset - np.argmax(win9)
        elif offset_grad_10 < grad_threshold and offset_val_10 > current_hr_max:
            end_of_test_idx = end_of_test_idx - 10 * offset - np.argmax(win10)
        elif offset_grad_11 < grad_threshold and offset_val_11 > current_hr_max:
            end_of_test_idx = end_of_test_idx - 11 * offset - np.argmax(win11)
        elif offset_grad_12 < grad_threshold and offset_val_12 > current_hr_max:
            end_of_test_idx = end_of_test_idx - 12 * offset - np.argmax(win12)
        else:
            break
    return round(len(time) * trim_coef) + end_of_test_idx

In [ ]:
def load_test_times(source):
    test_times = pd.DataFrame(columns=["testID", "load_start_time", "exhaustion_time"])
    times = pd.read_excel(source)
    load_start_time = times["load_start_time"].astype(str)
    load_start_time = pd.to_timedelta(load_start_time).dt.total_seconds().astype(int)
    exhaustion_time = times["exhaustion_time"].astype(str)
    exhaustion_time = pd.to_timedelta(exhaustion_time).dt.total_seconds().astype(int)
    test_times["testID"] = times["testID"]
    test_times["load_start_time"] = load_start_time
    test_times["exhaustion_time"] = exhaustion_time
    return test_times

In [ ]:
def trim_HR_signal(time, hr_signal, testID, exhaustion_times):
    detected_exhaustion_idx = detect_end_of_test(hr_signal, time)
    tolerance = 5000
    first_load = exhaustion_times.loc[
        exhaustion_times["testID"] == testID, "load_start_time"
    ].values[0]
    exhaustion_time = exhaustion_times.loc[
        exhaustion_times["testID"] == testID, "exhaustion_time"
    ].values[0]
    test_duration = exhaustion_time - first_load
    test_duration_samples = test_duration * 1000
    total_test_time = test_duration_samples + 2 * 60000
    time_total = np.linspace(0, total_test_time / 1000, total_test_time)
    first_load_computed = detected_exhaustion_idx - test_duration_samples
    if detected_exhaustion_idx + 60000 > len(hr_signal):
        last_val = hr_signal[-1]
        number_of_missing_samples = detected_exhaustion_idx + 60000 - len(hr_signal)
        if number_of_missing_samples > tolerance:
            raise ValueError(
                f"Recovery below 60s over {tolerance / 1000} seconds (missing {number_of_missing_samples / 1000}s)"
            )
        print(
            f"Recovery below 60s in {testID} (missing {(detected_exhaustion_idx + 60000 - len(hr_signal)) / 1000}s)"
        )
        missing_signal = [last_val] * number_of_missing_samples
        hr_signal = hr_signal[first_load_computed - 60000 : detected_exhaustion_idx + 60000]
        hr_signal = np.append(hr_signal, missing_signal)
    else:
        hr_signal = hr_signal[first_load_computed - 60000 : detected_exhaustion_idx + 60000]
    start_cutoff_idx = first_load_computed - 60000
    end_cutoff_idx = detected_exhaustion_idx + 60000
    return (
        time_total,
        hr_signal,
        start_cutoff_idx,
        end_cutoff_idx,
        detected_exhaustion_idx - start_cutoff_idx,
    )

In [ ]:
def trim_bbb_data(bbb_data, testID, exhaustion_times):
    first_load = exhaustion_times.loc[
        exhaustion_times["testID"] == testID, "load_start_time"
    ].values[0]
    exhaustion_time = exhaustion_times.loc[
        exhaustion_times["testID"] == testID, "exhaustion_time"
    ].values[0]
    dif = 0
    try:
        start = np.where(pd.to_timedelta(bbb_data["time"]).dt.total_seconds() <= first_load - 60)[0]
        if len(start) == 0:
            start = 0
        else:
            start = np.where(
                pd.to_timedelta(bbb_data["time"]).dt.total_seconds() <= first_load - 60
            )[0][-1]
        try:
            end = np.where(
                pd.to_timedelta(bbb_data["time"]).dt.total_seconds() >= exhaustion_time + 60
            )[0][0]
        except:
            end = bbb_data["time"].shape[0] - 1
            end_time = pd.to_timedelta(bbb_data.loc[end, "time"]).total_seconds()
            dif = end_time - exhaustion_time
    except:
        time = bbb_data["time"].str.split(":")
        time = time.apply(lambda row: list(map(int, row)))
        time = time.apply(lambda row: int(row[0] * 60) + int(row[1]))
        if len(np.where(time <= first_load - 60)[0]) > 1:
            start = np.where(time <= first_load - 60)[0][-1]
        elif len(np.where(time <= first_load - 60)[0]) > 0:
            start = np.where(time <= first_load - 60)[0][0]
        else:
            start = 0
        try:
            end = np.where(time >= exhaustion_time + 60)[0][0]
        except:
            end = bbb_data["time"].shape[0] - 1
            dif = time[end] - exhaustion_time
    if dif != 0 and 60 - dif > 5:
        raise ValueError(f"Recovery time below 60s in {testID} (found {dif} seconds)")
    elif dif != 0:
        print(f"Recovery time below 60s in {testID} (found {dif} seconds)")
    return bbb_data.loc[start:end]

## Input checks and processing

In [ ]:
def checkout_root(start):
    """Find the checkout without importing the Python 3.11 package."""
    start = Path(start).resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/ai_cpet_ots").is_dir():
            return candidate
    raise FileNotFoundError("Launch Jupyter from the checkout root or its notebooks directory.")


def configured_path(name, default):
    path = Path(os.environ.get(name) or default).expanduser()
    return (path if path.is_absolute() else PROJECT_ROOT / path).resolve()


def ordered_ids(frame):
    """Read caller-specified processing order; do not infer the historical cohort."""
    if list(frame.columns) != ["testID"] or frame.empty:
        raise ValueError("Participant list must contain only a nonempty testID column.")
    ids = frame["testID"].tolist()
    if any(
        not isinstance(value, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", value)
        for value in ids
    ):
        raise ValueError(
            "Use opaque string testID values containing letters, digits, underscores or hyphens."
        )
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate testID values in participant list.")
    return ids


def prepare_cohort(raw, cohort):
    ids = ordered_ids(pd.read_excel(raw / (cohort + "_participants.xlsx")))
    times = load_test_times(raw / (cohort + "_test_times.xlsx"))
    if times["testID"].duplicated().any() or times["testID"].isna().any():
        raise ValueError("Timing metadata must contain unique, nonmissing testID values.")
    missing = set(ids) - set(times["testID"])
    if missing:
        raise ValueError("Selected participants are missing timing metadata.")
    selected = times.set_index("testID").loc[ids]
    if not (selected["exhaustion_time"] > selected["load_start_time"]).all():
        raise ValueError("exhaustion_time must follow load_start_time.")
    clinical = pd.read_excel(raw / (cohort + "_clinical.xlsx"))
    cpet = pd.read_excel(raw / (cohort + "_cpet.xlsx"))
    if list(clinical.columns) != ["testID", "age", "height", "sex", "weight"]:
        raise ValueError("Clinical columns must be testID, age, height, sex, weight in that order.")
    if len(cpet.columns) != 12 or cpet.columns[0] != "testID":
        raise ValueError("CPET requires testID followed by the eleven ordered features.")
    for frame in [clinical, cpet]:
        if frame["testID"].isna().any() or frame["testID"].duplicated().any():
            raise ValueError("Clinical/CPET identifiers must be nonmissing and unique.")
        if not set(ids).issubset(set(frame["testID"])):
            raise ValueError("Selected participants are missing clinical/CPET data.")
    if not clinical["sex"].isin(["male", "female"]).all():
        raise ValueError("Clinical sex values must be male or female.")
    for test_id in ids:
        for folder, suffix in [(cohort + "_bbb", ".xlsx"), (cohort + "_ecg", ".adicht")]:
            path = raw / folder / (test_id + suffix)
            if not path.is_file() or raw not in path.resolve().parents:
                raise FileNotFoundError(
                    "Missing recording or recording resolves outside the raw directory."
                )
    return {"ids": ids, "times": times, "clinical": clinical, "cpet": cpet}


def legacy_bbb(frame, test_id, times):
    """Apply the original trimming and time-reset expressions unchanged."""
    expected = [
        "testID",
        "time",
        "VO2 ml/kg/min",
        "VO2",
        "VCO2",
        "VE",
        "VA",
        "Fb",
        "VT",
        "VD",
        "VD/VT",
        "PETCO2",
        "PETO2",
        "RER",
        "VE/VO2",
        "VE/VCO2",
    ]
    if list(frame.columns) != expected or not (frame["testID"] == test_id).all():
        raise ValueError(
            "BBB requires matching testID, time and the fourteen documented features in order."
        )
    frame = trim_bbb_data(frame, test_id, times).copy()
    try:
        bbb_time = pd.to_timedelta(frame["time"]).dt.total_seconds()
        offset = bbb_time.iloc[0]
        frame["time"] = bbb_time - offset
    except Exception:
        bbb_time = frame["time"].str.split(":")
        bbb_time = bbb_time.apply(lambda row: list(map(int, row)))
        bbb_time = bbb_time.apply(lambda row: int(row[0] * 60) + int(row[1]))
        offset = bbb_time.iloc[0]
        frame["time"] = bbb_time - offset
    if (
        frame.empty
        or not np.isfinite(frame["time"]).all()
        or not frame["time"].is_monotonic_increasing
    ):
        raise ValueError("Trimmed BBB time must be nonempty, finite and increasing.")
    return frame


def legacy_recording(recording, test_id, times):
    """Retain original signal processing, trimming, and channel layout."""
    time, hr_signal = compute_HR_signal(recording)
    _, _, start, end, detected = trim_HR_signal(time, hr_signal, test_id, times)
    ecg1, ecg2 = return_raw_ecg(recording)
    ecg1 = ecg1[start:end]
    ecg2 = ecg2[start:end]
    if len(ecg1) != len(ecg2):
        if len(ecg2) == 0:
            combined = ecg1
        else:
            raise ValueError("Beat-by-beat channels have different lengths.")
    else:
        combined = np.array([ecg1, ecg2])
    # Guard incompatible results outside the historical algorithm; never reshape.
    if np.ndim(combined) != 2 or combined.shape[0] != 2:
        raise ValueError(
            "Legacy single-channel output is one-dimensional and incompatible with the model loader; not exported."
        )
    if start < 0 or combined.shape[1] < 30000:
        raise ValueError(
            "Legacy cutoffs start before the recording or leave less than 30 seconds; not exported."
        )
    if not np.isfinite(combined).all():
        raise ValueError("Nonfinite beat-by-beat output; not exported.")
    return (
        combined,
        {
            "start_sample": int(start),
            "end_sample": int(end),
            "detected_sample": int(detected + start),
        },
        (time, hr_signal),
    )


def file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def run_workflow(raw, destination, plot=False, reader=None):
    """Read raw files only; create a new, complete cohort intersection per run."""
    raw, destination = Path(raw).resolve(), Path(destination).resolve()
    if raw == destination or raw in destination.parents or destination in raw.parents:
        raise ValueError(
            "Raw inputs and generated outputs must be separate, nonnested directories."
        )
    if destination.exists():
        raise FileExistsError(
            "Destination already exists. Choose a new destination; no files were overwritten."
        )
    cohorts = {name: prepare_cohort(raw, name) for name in ["control", "ots"]}
    if set(cohorts["control"]["ids"]) & set(cohorts["ots"]["ids"]):
        raise ValueError("Participant identifiers must be distinct across cohorts.")
    if reader is None:
        if sys.platform != "win32" or sys.version_info[:3] != (3, 7, 4):
            raise RuntimeError(
                "Raw recording processing requires the Windows/Python 3.7.4 legacy environment."
            )
        import adi

        reader = adi.read_file
    from importlib_metadata import distributions

    inputs = sorted(path for path in raw.rglob("*") if path.is_file())
    source_hashes = {str(path.relative_to(raw)): file_digest(path) for path in inputs}
    report = {
        "environment": {
            "python": sys.version,
            "platform": sys.platform,
            "packages": {d.metadata["Name"]: d.version for d in distributions()},
        },
        "parameters": {
            "sample_rate_hz": 1000,
            "context_seconds": 60,
            "recovery_tolerance_seconds": 5,
            "algorithms": "historical; see notebook functions",
        },
        "processing_order": {name: c["ids"] for name, c in cohorts.items()},
        "input_sha256": source_hashes,
        "notebook_sha256": file_digest(PROJECT_ROOT / "notebooks/preprocess.ipynb"),
        "results": [],
    }
    destination.mkdir(parents=True, exist_ok=False)
    try:
        for cohort, config in cohorts.items():
            for modality in ["bbb", "ecg"]:
                (destination / (cohort + "_" + modality)).mkdir()
            accepted = []
            for test_id in config["ids"]:
                result = {"cohort": cohort, "testID": test_id, "status": "failed"}
                try:
                    bbb = legacy_bbb(
                        pd.read_excel(raw / (cohort + "_bbb") / (test_id + ".xlsx")),
                        test_id,
                        config["times"],
                    )
                    recording = reader(str(raw / (cohort + "_ecg") / (test_id + ".adicht")))
                    combined, cutoffs, diagnostic = legacy_recording(
                        recording, test_id, config["times"]
                    )
                except Exception as error:
                    result["error"] = type(error).__name__ + ": " + str(error)
                    report["results"].append(result)
                    continue
                # Conversion succeeds for both modalities before either file is written.
                with (destination / (cohort + "_bbb") / (test_id + ".xlsx")).open("xb") as target:
                    bbb.to_excel(target, index=False, engine="openpyxl")
                with (destination / (cohort + "_ecg") / (test_id + ".npy")).open("xb") as target:
                    np.save(target, combined)
                accepted.append(test_id)
                result.update(
                    {"status": "success", "cutoffs": cutoffs, "shape": list(combined.shape)}
                )
                report["results"].append(result)
                if plot:
                    time, hr = diagnostic
                    figure, axis = plt.subplots()
                    axis.plot(time / 60, hr)
                    axis.set(
                        xlabel="Time (minutes)",
                        ylabel="Heart rate (beats/minute)",
                        title="Trimming diagnostic",
                    )
                    axis.axvline(cutoffs["start_sample"] / 60000, color="green")
                    axis.axvline(cutoffs["end_sample"] / 60000, color="red")
                    plt.show()
                    plt.close(figure)
            for table in ["clinical", "cpet"]:
                # Match the original row filtering: keep source table order.
                frame = config[table]
                frame = frame.loc[frame["testID"].isin(accepted)]
                with (destination / (cohort + "_" + table + ".xlsx")).open("xb") as target:
                    frame.to_excel(target, index=False, engine="openpyxl")
        report["source_files_unchanged"] = all(
            file_digest(raw / name) == digest for name, digest in source_hashes.items()
        )
        if not report["source_files_unchanged"]:
            raise RuntimeError("Raw inputs changed during the run; outputs cannot be validated.")
        report["output_sha256"] = {
            str(p.relative_to(destination)): file_digest(p)
            for p in sorted(destination.rglob("*"))
            if p.is_file()
        }
        report["complete"] = True
    finally:
        # A fatal write failure keeps the partial run, marks it incomplete, and propagates.
        report.setdefault("complete", False)
        with (destination / "run-report.json").open("x", encoding="utf-8") as target:
            json.dump(report, target, indent=2)
    print(
        "Preprocessing finished. Review run-report.json and verify subject alignment before modeling."
    )
    return report

In [ ]:
run_preprocessing = False
plot_diagnostics = False

PROJECT_ROOT = checkout_root(Path.cwd())
RAW_DATA_DIR = configured_path("AI_CPET_RAW_DATA_DIR", "data/raw")
PREPROCESSED_DIR = configured_path("AI_CPET_PREPROCESSED_DIR", "data/preprocessed")

In [ ]:
if run_preprocessing:
    run_report = run_workflow(RAW_DATA_DIR, PREPROCESSED_DIR, plot=plot_diagnostics)
else:
    print(
        "Preprocessing disabled. Configure private inputs and explicitly enable run_preprocessing."
    )